# 03 — Konsolidasi Anotasi (Mode Single-Annotator: AI-only)

**Mode**: Pilihan B — `data/gold/sample.csv` (label hasil AI-assisted) langsung di-treat sebagai gold final, tanpa multi-rater IAA.

**Konsekuensi metodologi**: Cohen's kappa dan Krippendorff's alpha **tidak dihitung** karena hanya 1 anotator (AI/Claude). Limitasi ini didokumentasikan eksplisit di `data/gold/agreement_report.md` dan harus dicantumkan di metodologi skripsi (BAB III).

**Input**:
- `data/gold/sample.csv` — sudah berisi label AI di kolom `sentiment` + `tox_*`

**Output**:
- `data/gold/gold_final.csv` — copy dari sample.csv dengan kolom `confidence_sentiment` ditambahkan (= `"ai_single"`)
- `data/gold/train.csv` — 70% stratified split
- `data/gold/test.csv` — 30% stratified split (held-out untuk notebook 06)
- `data/gold/agreement_report.md` — catatan single-annotator + distribusi label

**Untuk migrasi ke multi-annotator nanti** (Pilihan A): cukup buat `data/gold/raw_annotations/sample_*.csv` per anotator manusia, lalu re-run versi notebook 03 yang lama (commit history) yang memanggil `cohen_kappa_pairwise` + `krippendorff_alpha_multilabel`.

In [1]:
# Sel 1: Setup
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog
config = load_config('configs/experiment.yaml')
print_banner('03_annotation_review', config)
run_log = RunLog(notebook='03_annotation_review', config_path='configs/experiment.yaml')

GOLD_ROOT = Path(config['data']['gold_root'])
TOX_LABELS = list(config['labels']['toxicity_labels'])
SEED = int(config['seed'])

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 03_annotation_review
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: be03038
Started at: 2026-05-04T08:46:17+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: not-installed
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4


In [2]:
# Sel 2: Load sample.csv (single-annotator AI mode)
import pandas as pd

sample_path = GOLD_ROOT / 'sample.csv'
df_gold = pd.read_csv(sample_path)
print(f'Loaded: {sample_path} ({len(df_gold):,} baris × {len(df_gold.columns)} kolom)')

# Validasi
n_labeled = (df_gold['annotator_id'] == 'AI').sum()
n_unlabeled = (df_gold['annotator_id'].fillna('') == '').sum()
print(f'\nDilabel (AI)   : {n_labeled:,}')
print(f'Belum dilabel  : {n_unlabeled:,}')

if n_unlabeled > 0:
    msg = f'PERINGATAN: {n_unlabeled} baris belum dilabel — akan dieksklusi dari gold_final'
    print(f'\n[WARN] {msg}')
    run_log.add_warning(msg)

# Filter hanya yang sudah dilabel
df_gold = df_gold[df_gold['annotator_id'] == 'AI'].copy().reset_index(drop=True)
print(f'\nGold final akan berisi: {len(df_gold):,} baris')

Loaded: data\gold\sample.csv (8,002 baris × 24 kolom)

Dilabel (AI)   : 8,002
Belum dilabel  : 0

Gold final akan berisi: 8,002 baris


In [3]:
# Sel 3: Validasi distribusi label
print('=== Distribusi sentiment ===')
print(df_gold['sentiment'].value_counts())

print('\n=== Distribusi toxicity flags ===')
for lbl in TOX_LABELS:
    col = f'tox_{lbl}'
    if col in df_gold.columns:
        n = (df_gold[col].fillna(0).astype(float).astype(int) == 1).sum()
        print(f'  {col:24s}: {n:5,}')

print('\n=== is_dota_jargon dan is_ambiguous ===')
print(f'  is_dota_jargon=1 : {(df_gold["is_dota_jargon"].fillna(0).astype(float).astype(int) == 1).sum():,}')
print(f'  is_ambiguous=1   : {(df_gold["is_ambiguous"].fillna(0).astype(float).astype(int) == 1).sum():,}')

# Validasi minimum dataset
min_per_class = df_gold['sentiment'].value_counts().min()
if min_per_class < 30:
    msg = f'PERINGATAN: kelas terkecil hanya {min_per_class} sample (< 30) — train/test split mungkin tidak stabil'
    print(f'\n[WARN] {msg}')
    run_log.add_warning(msg)
else:
    print(f'\nKelas terkecil: {min_per_class} sample (>= 30, OK untuk stratified split)')

=== Distribusi sentiment ===
sentiment
positive    4516
neutral     3384
negative     102
Name: count, dtype: int64

=== Distribusi toxicity flags ===
  tox_toxic               :    68
  tox_severe_toxic        :     0
  tox_obscene             :    53
  tox_threat              :     0
  tox_insult              :    66
  tox_identity_hate       :     2

=== is_dota_jargon dan is_ambiguous ===
  is_dota_jargon=1 : 4,258
  is_ambiguous=1   : 3,064

Kelas terkecil: 102 sample (>= 30, OK untuk stratified split)


In [ ]:
# Sel 4: Dedup + tulis gold_final.csv
# Sample.csv punya beberapa baris dengan kunci (match_id, time, player_slot)
# yang sama (mis. 2 pesan di detik yang sama oleh player yang sama). Kita
# dedupe di sini supaya train/test split TIDAK overlap (data leak).
n_before = len(df_gold)
df_gold_dedup = df_gold.drop_duplicates(
    subset=['match_id', 'time', 'player_slot'], keep='first'
).reset_index(drop=True)
n_dropped = n_before - len(df_gold_dedup)
if n_dropped > 0:
    msg = f'Dedup: drop {n_dropped} duplicate keys (match_id+time+player_slot)'
    print(f'[INFO] {msg}')
    run_log.add_warning(msg)
print(f'Setelah dedup: {len(df_gold_dedup):,} baris unik')

# Tambah kolom confidence supaya schema kompatibel dengan downstream notebooks
gold = df_gold_dedup.copy()
gold['confidence_sentiment'] = 'ai_single'
for lbl in TOX_LABELS:
    gold[f'conf_tox_{lbl}'] = 'ai_single'
gold['n_annotators'] = 1

gold_path = GOLD_ROOT / 'gold_final.csv'
gold.to_csv(gold_path, index=False, encoding='utf-8')
print(f'Tertulis: {gold_path} ({len(gold):,} baris × {len(gold.columns)} kolom)')
run_log.add_output(gold_path)

In [5]:
# Sel 5: Tulis agreement_report.md (mode single-annotator)
import datetime as dt

n_total = len(gold)
sent_dist = gold['sentiment'].value_counts()
tox_counts = {lbl: int((gold[f'tox_{lbl}'].fillna(0).astype(float).astype(int) == 1).sum())
              for lbl in TOX_LABELS}
n_jargon = int((gold['is_dota_jargon'].fillna(0).astype(float).astype(int) == 1).sum())
n_ambig = int((gold['is_ambiguous'].fillna(0).astype(float).astype(int) == 1).sum())

report_lines = [
    '# Agreement Report — Gold-Standard Anotasi (Mode Single-Annotator)',
    '',
    f'Seed: {SEED}  |  Eksekusi: {dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}',
    f'Total sample: {n_total:,}',
    '',
    '## Mode anotasi: AI-only (Single-Annotator, Pilihan B)',
    '',
    'Sample diberi label oleh **1 anotator AI** (Claude, model claude-opus-4-7) menggunakan',
    'kombinasi rule-based matching + reasoning LLM. **Cohen\'s kappa dan Krippendorff\'s',
    'alpha tidak dapat dihitung** karena membutuhkan minimal 2 rater independen.',
    '',
    '### Implikasi metodologi (untuk skripsi BAB III):',
    '',
    '1. **Inter-annotator agreement** TIDAK dilaporkan untuk gold-test set.',
    '2. **Reliability** gold-test set bergantung sepenuhnya pada konsistensi 1 anotator AI.',
    '3. **Bias risk**: jika model evaluasi (BERT/RoBERTa/DistilBERT) punya bias representasi',
    '   yang mirip dengan anotator AI, hasil F1 mungkin over-optimistic. Gunakan',
    '   `tox_identity_hate` dan kolom `is_ambiguous` sebagai proxy untuk audit.',
    '4. **Mitigasi**: 38% sample ditandai `is_ambiguous=1` — ini bisa di-eksklusi atau',
    '   di-evaluasi terpisah di notebook 06 untuk subgroup analysis.',
    '',
    '### Untuk validitas skripsi yang lebih kuat:',
    '',
    'Direkomendasikan migrasi ke **Pilihan A (multi-annotator)** sebelum publikasi —',
    'tim peneliti (Daniel/Dhitan/Aldiaz) label minimal 200 sample bersama, hitung kappa',
    'antara AI dan masing-masing anotator manusia. Threshold publikasi: kappa pair-wise',
    'AI-vs-human >= 0.6 untuk sentimen.',
    '',
    '## Distribusi label gold_final',
    '',
    '### Sentiment',
    '',
    '| Label | n | % |',
    '|-------|---|---|',
]
for s, n in sent_dist.items():
    report_lines.append(f'| {s} | {n:,} | {100*n/n_total:.1f}% |')

report_lines += [
    '',
    '### Toxicity (per-label, jumlah baris dengan label = 1)',
    '',
    '| Label | n | % |',
    '|-------|---|---|',
]
for lbl, n in tox_counts.items():
    report_lines.append(f'| {lbl} | {n} | {100*n/n_total:.2f}% |')

report_lines += [
    '',
    '### Markers',
    '',
    f'| Marker | n | % |',
    f'|--------|---|---|',
    f'| is_dota_jargon=1 | {n_jargon:,} | {100*n_jargon/n_total:.1f}% |',
    f'| is_ambiguous=1 | {n_ambig:,} | {100*n_ambig/n_total:.1f}% |',
    '',
    '## Catatan untuk reviewer',
    '',
    'Sample dengan `is_ambiguous=1` adalah baris yang anotator AI tidak yakin (mis. sarcasm',
    'tidak terdeteksi, frasa multi-bahasa, atau token tidak dikenal seperti "gwr"). Saran:',
    'eksklusi dari uji metrik utama atau lakukan analisis terpisah.',
]

report_path = GOLD_ROOT / 'agreement_report.md'
report_path.write_text('\n'.join(report_lines), encoding='utf-8')
print(f'Tertulis: {report_path}')
run_log.add_output(report_path)

Tertulis: data\gold\agreement_report.md


In [ ]:
# Sel 6: Split train (70%) / test (30%) stratified by sentiment + refresh manifest
import hashlib
import yaml
from src.gold.agreement import split_train_test

train, test = split_train_test(gold, test_size=float(config['evaluation']['test_split_ratio']), seed=SEED)
train_path = GOLD_ROOT / 'train.csv'
test_path = GOLD_ROOT / 'test.csv'
train.to_csv(train_path, index=False, encoding='utf-8')
test.to_csv(test_path, index=False, encoding='utf-8')

# Sanity check: tidak ada overlap key antara train dan test
train_keys = set(zip(train['match_id'], train['time'], train['player_slot']))
test_keys = set(zip(test['match_id'], test['time'], test['player_slot']))
overlap = train_keys & test_keys
if overlap:
    msg = f'CRITICAL: train ∩ test overlap {len(overlap)} keys — DATA LEAK!'
    print(f'[ERROR] {msg}')
    run_log.add_warning(msg)
    raise RuntimeError(msg)
print(f'train: {len(train):,} → {train_path}')
print(f'test : {len(test):,} → {test_path}')
print(f'overlap train ∩ test: 0 (clean)')

# Refresh hash di manifest: train, test, dan sample (yang termodifikasi pasca anotasi)
sample_path = GOLD_ROOT / 'sample.csv'
h_train = hashlib.sha256(train_path.read_bytes()).hexdigest()
h_test = hashlib.sha256(test_path.read_bytes()).hexdigest()
h_sample = hashlib.sha256(sample_path.read_bytes()).hexdigest()

cfg_path = Path('configs/experiment.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
cfg['data']['gold_split_hash']['train'] = h_train
cfg['data']['gold_split_hash']['test'] = h_test
cfg['data']['gold_split_hash']['sample'] = h_sample
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(f'\ntrain  sha256: {h_train}')
print(f'test   sha256: {h_test}')
print(f'sample sha256: {h_sample}')
run_log.add_output(train_path)
run_log.add_output(test_path)
run_log.add_output(cfg_path)

In [7]:
# Sel 8: Run log
run_log.save('reports/run_log.csv')

[run_log] 03_annotation_review → 1.63s, 5 outputs, 0 warnings → reports\run_log.csv
